# Scraping X via API Resmi: Kebijakan WFH ASN Hari Jumat

Notebook ini mengambil data dari X menggunakan **API v2 resmi** (lewat `tweepy`), lalu menganalisis sentimennya.

### Kelebihan vs snscrape
- ✅ Stabil & legal, tidak diblokir seperti scraping langsung di Colab
- ✅ Data terstruktur (metrik like/retweet/reply tersedia)

### Kekurangan
- ⚠️ Butuh **Bearer Token** (daftar di https://developer.x.com)
- ⚠️ **Hanya 7 hari terakhir** (endpoint recent search)
- ⚠️ **Kuota tier gratis sangat kecil** (~100 tweet/bulan). Untuk volume besar perlu tier Basic (berbayar).

## 1. Cara mendapatkan Bearer Token

1. Buka https://developer.x.com lalu daftar / login akun developer.
2. Buat sebuah **Project** + **App**.
3. Di halaman App, buka tab **Keys and tokens**.
4. Salin **Bearer Token**.

Token ini yang akan dimasukkan pada sel berikutnya.

## 2. Install dependency

In [ ]:
!pip install -q tweepy
print('tweepy terpasang.')

## 3. Masukkan Bearer Token

Token diminta lewat input tersembunyi agar tidak ikut tersimpan di notebook.

In [ ]:
from getpass import getpass

BEARER_TOKEN = getpass('Tempel Bearer Token kamu lalu tekan Enter: ')
print('Token diterima.' if BEARER_TOKEN else 'Token kosong!')

## 4. Konfigurasi & jalankan scraping

Catatan: endpoint recent search hanya mencakup 7 hari terakhir. `MAX_TWEETS` di-set wajar (500); naikkan jika kuota tier kamu mendukung.

In [ ]:
import tweepy
import pandas as pd

MAX_TWEETS = 500
OUTPUT_FILE = 'tweets_wfh_asn_api.csv'

QUERY = (
    '("WFH ASN" OR "WFH PNS" OR "ASN WFH" OR "work from home ASN" '
    'OR (ASN Jumat WFH) OR (ASN "kerja dari rumah")) '
    'lang:id -is:retweet'
)

client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

rows = []
paginator = tweepy.Paginator(
    client.search_recent_tweets,
    query=QUERY,
    tweet_fields=['created_at', 'author_id', 'public_metrics', 'lang'],
    max_results=100,
)

for tweet in paginator.flatten(limit=MAX_TWEETS):
    m = tweet.public_metrics or {}
    rows.append({
        'id': tweet.id,
        'created_at': tweet.created_at,
        'author_id': tweet.author_id,
        'lang': tweet.lang,
        'likes': m.get('like_count', 0),
        'retweets': m.get('retweet_count', 0),
        'replies': m.get('reply_count', 0),
        'text': tweet.text.replace('\n', ' '),
    })
    if len(rows) % 100 == 0:
        print(f'  ... {len(rows)} tweet terkumpul')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'\nSelesai. {len(df)} tweet tersimpan di {OUTPUT_FILE}')
df.head()

## 5. Unduh CSV

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## 6. (Opsional) Analisis sentimen IndoBERT

In [ ]:
!pip install -q transformers torch

import re
from transformers import pipeline

LABEL_MAP = {'LABEL_0': 'positif', 'LABEL_1': 'netral', 'LABEL_2': 'negatif'}

def bersihkan(t):
    t = re.sub(r'http\S+|www\.\S+', '', str(t))
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_text'] = df['text'].apply(bersihkan)
df = df[df['clean_text'].str.len() > 3].reset_index(drop=True)

clf = pipeline('sentiment-analysis',
               model='mdhugol/indonesia-bert-sentiment-classification',
               truncation=True, max_length=256)

hasil = clf(df['clean_text'].tolist())
df['sentimen'] = [LABEL_MAP.get(r['label'], r['label']) for r in hasil]
df['skor'] = [round(r['score'], 3) for r in hasil]
df.to_csv('tweets_wfh_asn_sentimen.csv', index=False, encoding='utf-8')
print(df['sentimen'].value_counts())
df[['text', 'sentimen', 'skor']].head(10)